## **Aim**
To implement a program that analyzes firewall logs and identifies blocked connections and repeated unauthorized access attempts.

## **Algorithm**
**Step 1:** Import `re`, `collections.Counter`, `json`, and `datetime` libraries.

**Step 2:** Create a simulated firewall log (JSON) with fields: timestamp, source_ip, dest_ip, source_port, dest_port, protocol, action (allow/deny/drop), rule_name.

**Step 3:** Parse the log and extract all denied/dropped connections.

**Step 4:** Group denied connections by source IP and destination IP.

**Step 5:** Identify repeated unauthorized access attempts from the same source IP.

**Step 6:** Detect port scanning patterns (many ports, same destination).

**Step 7:** Generate a report of blocked connections and suspicious patterns.

In [1]:
import re
import json
import shutil
from collections import Counter, defaultdict
from datetime import datetime, timedelta

def create_sample_firewall_log(log_file):
    now = datetime.now()
    base = now - timedelta(hours=4)
    
    events = []
    
    # Normal allowed traffic
    for i in range(100):
        events.append({
            "timestamp": (base + timedelta(minutes=i*2)).isoformat(),
            "source_ip": "192.168.1.50",
            "dest_ip": "8.8.8.8",
            "source_port": 1024 + i,
            "dest_port": 53,
            "protocol": "UDP",
            "action": "ALLOW",
            "rule": "DNS_OUTBOUND"
        })
    
    for i in range(45):
        events.append({
            "timestamp": (base + timedelta(minutes=i*4)).isoformat(),
            "source_ip": "192.168.1.200",
            "dest_ip": "192.168.1.10",
            "source_port": 40000 + i,
            "dest_port": 443,
            "protocol": "TCP",
            "action": "ALLOW",
            "rule": "HTTPS_INBOUND"
        })
    
    # SSH brute force from 10.0.0.100
    for i in range(15):
        events.append({
            "timestamp": (base + timedelta(minutes=i*10)).isoformat(),
            "source_ip": "10.0.0.100",
            "dest_ip": "192.168.1.10",
            "source_port": 50000 + i,
            "dest_port": 22,
            "protocol": "TCP",
            "action": "DENY",
            "rule": "BLOCK_SSH_EXTERNAL"
        })
    
    # RDP brute force from 10.0.0.100
    for i in range(10):
        events.append({
            "timestamp": (base + timedelta(minutes=10 + i*15)).isoformat(),
            "source_ip": "10.0.0.100",
            "dest_ip": "192.168.1.10",
            "source_port": 51000 + i,
            "dest_port": 3389,
            "protocol": "TCP",
            "action": "DENY",
            "rule": "BLOCK_RDP_EXTERNAL"
        })
    
    # Port scan from 10.0.0.100
    scan_ports = [21, 22, 23, 25, 80, 135, 443, 3389]
    for i, port in enumerate(scan_ports):
        events.append({
            "timestamp": (base + timedelta(minutes=30 + i*5)).isoformat(),
            "source_ip": "10.0.0.100",
            "dest_ip": "192.168.1.10",
            "source_port": 52000 + i,
            "dest_port": port,
            "protocol": "TCP",
            "action": "DENY",
            "rule": "BLOCK_SCAN_PORTS"
        })
    
    # SSH brute force from 172.16.0.50
    for i in range(12):
        events.append({
            "timestamp": (base + timedelta(minutes=100 + i*8)).isoformat(),
            "source_ip": "172.16.0.50",
            "dest_ip": "192.168.1.10",
            "source_port": 60000 + i,
            "dest_port": 22,
            "protocol": "TCP",
            "action": "DENY",
            "rule": "BLOCK_SSH_EXTERNAL"
        })
    
    # RDP brute force from 172.16.0.50
    for i in range(8):
        events.append({
            "timestamp": (base + timedelta(minutes=100 + i*12)).isoformat(),
            "source_ip": "172.16.0.50",
            "dest_ip": "192.168.1.10",
            "source_port": 61000 + i,
            "dest_port": 3389,
            "protocol": "TCP",
            "action": "DENY",
            "rule": "BLOCK_RDP_EXTERNAL"
        })
    
    # External attackers
    for i in range(5):
        events.append({
            "timestamp": (base + timedelta(hours=i)).isoformat(),
            "source_ip": "203.0.113.45",
            "dest_ip": "192.168.1.10",
            "source_port": 70000 + i,
            "dest_port": 22,
            "protocol": "TCP",
            "action": "DENY",
            "rule": "BLOCK_SSH_EXTERNAL"
        })
    
    for i in range(3):
        events.append({
            "timestamp": (base + timedelta(hours=i*2)).isoformat(),
            "source_ip": "198.51.100.23",
            "dest_ip": "192.168.1.10",
            "source_port": 80000 + i,
            "dest_port": 445,
            "protocol": "TCP",
            "action": "DENY",
            "rule": "BLOCK_SMB_EXTERNAL"
        })
    
    # Some internal noise
    for i in range(2):
        events.append({
            "timestamp": (base + timedelta(minutes=180 + i*30)).isoformat(),
            "source_ip": "192.168.1.50",
            "dest_ip": "192.168.1.10",
            "source_port": 90000 + i,
            "dest_port": 22,
            "protocol": "TCP",
            "action": "DENY",
            "rule": "BLOCK_SSH_INTERNAL"
        })
    
    with open(log_file, "w") as f:
        json.dump(events, f, indent=2)

def analyze_firewall_log(log_file):
    with open(log_file, "r") as f:
        events = json.load(f)
    
    denied = [e for e in events if e["action"] in ("DENY", "DROP")]
    allowed = [e for e in events if e["action"] == "ALLOW"]
    
    # Count by source IP
    src_ip_counts = Counter(e["source_ip"] for e in denied)
    
    # Count by destination port
    dest_port_counts = Counter(e["dest_port"] for e in denied)
    
    # Group by source->dest->port
    attack_groups = defaultdict(list)
    for e in denied:
        key = (e["source_ip"], e["dest_ip"], e["dest_port"])
        attack_groups[key].append(e)
    
    # Detect repeated attempts
    repeated_attacks = []
    for (src, dst, port), attempts in attack_groups.items():
        if len(attempts) >= 5:
            attempts.sort(key=lambda x: x["timestamp"])
            first = datetime.fromisoformat(attempts[0]["timestamp"])
            last = datetime.fromisoformat(attempts[-1]["timestamp"])
            time_span = last - first
            
            port_names = {22: "SSH", 23: "Telnet", 25: "SMTP", 53: "DNS",
                        80: "HTTP", 135: "RPC", 443: "HTTPS", 445: "SMB",
                        1433: "SQL", 3306: "MySQL", 3389: "RDP", 5432: "PostgreSQL",
                        5900: "VNC", 6379: "Redis", 27017: "MongoDB"}
            service = port_names.get(port, f"Port {port}")
            
            if port in [22, 3389] and len(attempts) >= 10:
                severity = "CRITICAL"
                pattern = f"{service} brute force"
            elif port in [22, 3389]:
                severity = "HIGH"
                pattern = f"{service} brute force"
            elif len(attempts) >= 8:
                severity = "HIGH"
                pattern = "Repeated access attempts"
            else:
                severity = "MEDIUM"
                pattern = "Repeated access attempts"
            
            repeated_attacks.append({
                "src": src, "dst": dst, "port": port,
                "service": service, "count": len(attempts),
                "time_span": str(time_span), "severity": severity,
                "pattern": pattern
            })
    
    # Detect port scans
    port_scans = []
    src_dest_ports = defaultdict(set)
    for e in denied:
        src_dest_ports[(e["source_ip"], e["dest_ip"])].add(e["dest_port"])
    
    for (src, dst), ports in src_dest_ports.items():
        if len(ports) >= 5:
            port_scans.append({
                "src": src, "dst": dst,
                "ports": sorted(ports),
                "port_count": len(ports)
            })
    
    return {
        "total": len(events),
        "allowed": len(allowed),
        "denied": len(denied),
        "src_ip_counts": src_ip_counts,
        "dest_port_counts": dest_port_counts,
        "repeated_attacks": repeated_attacks,
        "port_scans": port_scans
    }

def main():
    log_file = "firewall_log.json"
    create_sample_firewall_log(log_file)
    
    print("Analyzing firewall log...")
    results = analyze_firewall_log(log_file)
    
    print(f"\n{'='*60}")
    print(f"FIREWALL LOG ANALYSIS REPORT")
    print(f"{'='*60}")
    print(f"Total log entries: {results['total']}")
    print(f"Allowed connections: {results['allowed']}")
    print(f"Denied/Dropped connections: {results['denied']}")
    
    print(f"\n--- TOP DENIED SOURCE IPs ---")
    for i, (ip, count) in enumerate(results['src_ip_counts'].most_common(10), 1):
        print(f"{i}. {ip}: {count} denied")
    
    print(f"\n--- TOP DENIED DESTINATION PORTS ---")
    port_names = {22: "SSH", 23: "Telnet", 25: "SMTP", 53: "DNS",
                80: "HTTP", 135: "RPC", 443: "HTTPS", 445: "SMB",
                1433: "SQL", 3306: "MySQL", 3389: "RDP", 5432: "PostgreSQL",
                5900: "VNC", 6379: "Redis", 27017: "MongoDB"}
    for i, (port, count) in enumerate(results['dest_port_counts'].most_common(10), 1):
        service = port_names.get(port, f"Port {port}")
        print(f"{i}. Port {port} ({service}): {count} denied")
    
    print(f"\n--- REPEATED UNAUTHORIZED ACCESS ATTEMPTS ---")
    
    for i, attack in enumerate(results['repeated_attacks'], 1):
        print(f"\n{i}. [{attack['severity']}] {attack['src']} -> {attack['dst']} (Port {attack['port']})")
        print(f"   Attempts: {attack['count']} | Time span: {attack['time_span']}")
        print(f"   Pattern: {attack['pattern']}")
    
    print(f"\n--- PORT SCAN DETECTION ---")
    for i, scan in enumerate(results['port_scans'], 1):
        print(f"\n{i}. {scan['src']} scanned {scan['port_count']} ports on {scan['dst']}")
        print(f"   Ports: {', '.join(map(str, scan['ports']))}")
        print(f"   Classification: Port scan")
    
    critical = sum(1 for a in results['repeated_attacks'] if a['severity'] == 'CRITICAL')
    high = sum(1 for a in results['repeated_attacks'] if a['severity'] == 'HIGH')
    print(f"\n--- SUMMARY ---")
    print(f"  Critical alerts: {critical}")
    print(f"  High alerts: {high}")
    print(f"  Total blocked IPs: {len(results['src_ip_counts'])}")

if __name__ == "__main__":
    main()

Analyzing firewall log...

FIREWALL LOG ANALYSIS REPORT
Total log entries: 208
Allowed connections: 145
Denied/Dropped connections: 63

--- TOP DENIED SOURCE IPs ---
1. 10.0.0.100: 33 denied
2. 172.16.0.50: 20 denied
3. 203.0.113.45: 5 denied
4. 198.51.100.23: 3 denied
5. 192.168.1.50: 2 denied

--- TOP DENIED DESTINATION PORTS ---
1. Port 22 (SSH): 35 denied
2. Port 3389 (RDP): 19 denied
3. Port 445 (SMB): 3 denied
4. Port 21 (Port 21): 1 denied
5. Port 23 (Telnet): 1 denied
6. Port 25 (SMTP): 1 denied
7. Port 80 (HTTP): 1 denied
8. Port 135 (RPC): 1 denied
9. Port 443 (HTTPS): 1 denied

--- REPEATED UNAUTHORIZED ACCESS ATTEMPTS ---

1. [CRITICAL] 10.0.0.100 -> 192.168.1.10 (Port 22)
   Attempts: 16 | Time span: 2:20:00
   Pattern: SSH brute force

2. [CRITICAL] 10.0.0.100 -> 192.168.1.10 (Port 3389)
   Attempts: 11 | Time span: 2:15:00
   Pattern: RDP brute force

3. [CRITICAL] 172.16.0.50 -> 192.168.1.10 (Port 22)
   Attempts: 12 | Time span: 1:28:00
   Pattern: SSH brute force

4. 

## **Result**
This the program successfully analyzes firewall logs and identifies blocked connections and repeated unauthorized access attempts.